# GPT-Style Language Model From Scratch (PyTorch)

This project implements a character-level GPT-style Transformer trained on the OpenWebText dataset.

The model learns language patterns directly from raw internet text and generates coherent paragraphs without instruction tuning.

### Implemented Concepts
- Multi-Head Self Attention
- Positional Embeddings
- Autoregressive Text Generation
- Top-K Sampling
- GPU Training with PyTorch



In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset
from collections import deque
import string
import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

# safe for RTX 2050
batch_size = 16
block_size = 128
max_iters = 20000
eval_interval = 1000
eval_iters = 200
learning_rate = 1e-4

n_embd = 256
n_head = 4
n_layers = 6
dropout = 0.2

device: cuda


In [2]:
chars = list(string.printable)
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

def encode(s):
    return [stoi.get(c, stoi[' ']) for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])


In [3]:
dataset = load_dataset("Skylion007/openwebtext", split="train", streaming=True)
data_iter = iter(dataset)

buffer = deque(maxlen=5_000_000)  # rolling memory buffer

def refill_buffer(min_size=1_000_000):
    while len(buffer) < min_size:
        try:
            text = next(data_iter)["text"]
            buffer.extend(text + "\n")
        except StopIteration:
            break

refill_buffer()

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

In [4]:
def get_batch(split):
    refill_buffer()

    data = list(buffer)
    ix = torch.randint(len(data)-block_size-1, (batch_size,))

    x = torch.stack([
        torch.tensor(encode(''.join(data[i:i+block_size])), dtype=torch.long)
        for i in ix
    ])

    y = torch.stack([
        torch.tensor(encode(''.join(data[i+1:i+block_size+1])), dtype=torch.long)
        for i in ix
    ])

    return x.to(device), y.to(device)

In [5]:
@torch.no_grad() # make sure pytorch does not use gradient descent to improve speed 
def estimate_loss():
    out = {}
    model.eval() # Put the model in evaluation mode
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() #Put model in training mode updating weights and biases
    return out
    

In [6]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size*num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd,4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd,n_embd),
            nn.Dropout(dropout),
        )
    def forward(self,x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd//n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


In [7]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size,n_embd)
        self.position_embedding = nn.Embedding(block_size,n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd,vocab_size)

    def forward(self,idx,targets=None):
        B,T = idx.shape
        tok = self.token_embedding(idx)
        pos = self.position_embedding(torch.arange(T,device=device))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            return logits, None

        B,T,C = logits.shape
        loss = F.cross_entropy(logits.view(B*T,C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.7, top_k=40):
        for _ in range(max_new_tokens):
    
            idx_cond = idx[:, -block_size:]
    
            logits, _ = self(idx_cond)   # ← FIX HERE
    
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
    
            v, ix = torch.topk(probs, top_k)
            probs = torch.zeros_like(probs).scatter_(1, ix, v)
            probs = probs / probs.sum(dim=-1, keepdim=True)
    
            next_token = torch.multinomial(probs, 1)
            idx = torch.cat((idx, next_token), dim=1)
    
        return idx


In [8]:
model = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scaler = torch.cuda.amp.GradScaler()

@torch.no_grad()
def estimate_loss():
    model.eval()
    out={}
    for split in ['train']:
        losses=torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y=get_batch(split)
            with torch.cuda.amp.autocast():
                _,loss=model(X,Y)
            losses[k]=loss.item()
        out[split]=losses.mean()
    model.train()
    return out

C:\Users\ACER\AppData\Local\Temp\ipykernel_16136\683996256.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


## Model trained for ~20,000 iterations on OpenWebText using NVIDIA RTX 2050 GPU.


In [ ]:
for iter in range(max_iters):

    if iter % eval_interval==0:
        print(iter, estimate_loss())

    xb,yb=get_batch('train')

    with torch.cuda.amp.autocast():
        _,loss=model(xb,yb)

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    if iter % 2000 == 0:
        torch.save(model.state_dict(), f"checkpoint_{iter}.pt")


# Load pretrained model

In [9]:
model = GPTLanguageModel().to(device)

ckpt = torch.load("checkpoint_18000.pt", map_location=device)

try:
    model.load_state_dict(ckpt["model_state_dict"])
except:
    model.load_state_dict(ckpt)

model.eval()
print("Model loaded successfully ✅")


Model loaded successfully ✅


In [13]:
def generate_unprompted(tokens=1000):
    model.eval()

    # start from newline instead of zero token
    start = "The"
    context = torch.tensor(encode(start), dtype=torch.long, device=device).unsqueeze(0)

    out = model.generate(context, tokens, temperature=0.6, top_k=40)
    text = decode(out[0].tolist())

    print(text)


In [14]:
generate_unprompted()


The Catast Regressions and contents on the first in the virtual versual country, and they continue to grave the US probabuse setting studies and political policy and one of comparable of the evidence of the most of website to see manage a former than in the best of the some shown whether the United States in the time is that on their particials of a cent of the first since the expectation by the minimum-wage discussion in the policy of the great in a home of the moon of the first take form, the first particularly has been been the structure with latest a just better started in the second to be a frame of only seek only to hange problem factors and we event and containing the came be in a similar of a mayor space a for matching of someone saying it coming.

A particular state to a political finding from side addition accept by studies that IQ the latest effects of the panel compared in the final to be should be seen the possible smally first because the paper of minimum wage the and com